# Vishwakarma Institute of Technology, Pune
## Course: CI3201 - Deep Learning
### Lab Practical 4: LSTM-Based Model for Time-Series Forecasting (Stock Price Prediction)

---

## 1. Objective & Theoretical Background

### 1.1 Objective
Develop an LSTM-based model for time-series forecasting. We will use historical Apple Stock (AAPL) prices to predict the future closing price based on the previous $N$ days of data.

### 1.2 Mathematical Foundations of LSTM
Long Short-Term Memory (LSTM) networks overcome the vanishing gradient problem in traditional RNNs using gating mechanisms:
1. **Forget Gate**: $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$ - Decides what information to discard.
2. **Input Gate**: $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$ - Decides what new info to store.
3. **Candidate Cell State**: $\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$
4. **Update Cell State**: $C_t = f_t * C_{t-1} + i_t * \tilde{C}_t$
5. **Output Gate**: $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$
6. **Hidden State**: $h_t = o_t * \tanh(C_t)$

For Time-Series Forecasting, the model outputs a continuous value, so we use **Mean Squared Error (MSE)** as our loss function.

In [ ]:
# Install yfinance to fetch historical stock data (Uncomment if not installed)
# !pip install yfinance

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Activation
from keras.optimizers import RMSprop
from keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Fetch AAPL Stock Data from Yahoo Finance
print("Fetching Data...")
df = yf.download('AAPL', start='2015-01-01', end='2024-01-01')
df.reset_index(inplace=True)

# Save to Datasets directory for future reference
df.to_csv('Datasets/AAPL_stock_prices.csv', index=False)

print("Data Shape:", df.shape)
df.head()

In [ ]:
# 2. Visualize the Historical Closing Price
plt.figure(figsize=(16, 6))
plt.plot(df['Date'], df['Close'], color='blue', label='AAPL Closing Price')
plt.title('Apple (AAPL) Historical Stock Closing Price')
plt.xlabel('Date')
plt.ylabel('Closing Price (USD)')
plt.legend()
plt.grid(True)
plt.show()

## 2. Data Preprocessing

- **Scaling**: LSTM is highly sensitive to the magnitude of data. We use `MinMaxScaler` to scale the prices to `(0, 1)`.
- **Sequence Generation**: We use the past 60 days (`time_steps = 60`) of data to predict the 61st day's price.

In [ ]:
# Extract Closing Prices
data = df.filter(['Close']).values

# Scale Data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# Define Sequence Length (Time Steps)
time_steps = 60

# Generate Sequences
X = []
Y = []
for i in range(time_steps, len(scaled_data)):
    X.append(scaled_data[i-time_steps:i, 0])
    Y.append(scaled_data[i, 0])
    
X, Y = np.array(X), np.array(Y)

# Reshape X for LSTM [samples, time_steps, features]
X = np.reshape(X, (X.shape[0], X.shape[1], 1))

# Chronological Train-Test Split (80% Train, 20% Test)
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
Y_train, Y_test = Y[:train_size], Y[train_size:]

print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)
print("X_test shape:", X_test.shape)

## 3. Build the LSTM Architecture using Keras Functional API

We construct the LSTM network using the **Keras Functional API** (similar to the SMS Spam example provided in class):
1. **Input Layer**: Shape = `[time_steps, 1]`
2. **LSTM Layer**: 64 units to capture temporal dependencies.
3. **Dropout**: 20% rate to prevent overfitting.
4. **Dense Output**: 1 unit for the continuous price prediction.

In [ ]:
def build_lstm_model():
    # Define Input Layer
    inputs = Input(name='inputs', shape=(time_steps, 1))
    
    # LSTM Layer
    layer = LSTM(64, return_sequences=False, name='lstm_1')(inputs)
    
    # Regularization
    layer = Dropout(0.2, name='dropout_1')(layer)
    
    # Output Layer (Linear Activation for Regression)
    layer = Dense(1, name='out_layer')(layer)
    
    model = Model(inputs=inputs, outputs=layer)
    return model

model = build_lstm_model()
model.summary()

# Compile Model
model.compile(loss='mean_squared_error', optimizer=RMSprop(learning_rate=0.001))

## 4. Train the Model with Early Stopping
We apply `EarlyStopping` to halt training when validation loss stops improving, restoring the best model weights.

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(X_train, Y_train,
                    epochs=50,
                    batch_size=32,
                    validation_split=0.1,
                    callbacks=[early_stop],
                    verbose=1)

## 5. Model Evaluation & Visualization

In [ ]:
# Plot Training vs Validation Loss
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss (MSE) over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.legend()
plt.show()

# Make Predictions on Test Set
predictions = model.predict(X_test)

# Inverse transform to get actual USD values
predictions_usd = scaler.inverse_transform(predictions)
Y_test_usd = scaler.inverse_transform(Y_test.reshape(-1, 1))

# Calculate Root Mean Squared Error (RMSE)
rmse = math.sqrt(mean_squared_error(Y_test_usd, predictions_usd))
print(f"Test RMSE: ${rmse:.2f}")

In [ ]:
# Plot Actual vs Predicted Prices
# We need to align the dates for the test set
test_data_start_idx = len(df) - len(Y_test)
test_dates = df['Date'].iloc[test_data_start_idx:].values

plt.figure(figsize=(16, 6))
plt.plot(test_dates, Y_test_usd, color='blue', label='Actual Apple Stock Price')
plt.plot(test_dates, predictions_usd, color='red', label='Predicted Apple Stock Price')
plt.title('Apple (AAPL) Stock Price Prediction using LSTM')
plt.xlabel('Date')
plt.ylabel('Closing Price (USD)')
plt.legend()
plt.grid(True)
plt.show()